In [2]:
import os
import pandas as pd
import numpy as np
import json
import systeme as sys
import Allocation
from simulation import simulation

In [5]:
verbose = 1
ds = "g3"
scenarios_path_prefixe = f"scenarios/{ds}/"
solutions_path_prefixe = f"solution/{ds}_best/"
with open(f'fms/5C14R5F.json', 'r') as json_file: #5C14R5F #3C7R5F
    dic = json.load(json_file)
system = sys.systeme(dic)
for scenario_path, solution_path in zip(sorted(os.listdir(scenarios_path_prefixe), key= lambda k : int(k[1:].split(".csv")[0])), sorted(os.listdir(solutions_path_prefixe), key= lambda k : int(k.split(".csv")[0].split("_s")[-1]))):
    if verbose > 0:
        print(f"scenario : {scenario_path}",end="\r")
    #initialisation de solution systeme et scenario
    solution = pd.read_csv(solutions_path_prefixe+solution_path, sep=";", index_col=None, header=None).iloc[:,1:-1]
    scenario = pd.DataFrame(np.nan_to_num(pd.read_csv(scenarios_path_prefixe+scenario_path,header=None, index_col=None, sep=";"), nan=0)).astype(int).iloc[:,:-1]
    sim = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))], saving=True)
    logs = sim.get_logs()
    [df.to_csv(f"generated_data/{ds}/single_label/singlelabel_data_{scenario_path.split(".")[0]}_cell_{i+1}.csv", sep=";", index=False) for df,i in zip(logs, range(len(logs)))]